In [1]:
text = '''Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on taskspecific datasets. We demonstrate that language models begin to learn these tasks without any explicit supervision when trained on a new dataset of millions of webpages called WebText. When conditioned on a document plus questions, the answers generated by the language model reach 55 F1 on the CoQA dataset - matching or exceeding the performance of 3 out of 4 baseline systems without using the 127,000+ training examples. The capacity of the language model is essential to the success of zero-shot task transfer and increasing it improves performance in a log-linear fashion across tasks. Our largest model, GPT-2, is a 1.5B parameter Transformer that achieves state of the art results on 7 out of 8 tested language modeling datasets in a zero-shot setting but still underfits WebText. Samples from the model reflect these improvements and contain coherent paragraphs of text. These findings suggest a promising path towards building language processing systems which learn to perform tasks from their naturally occurring demonstrations. ☑️'''
tokens = text.encode("utf-8") # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0....255 for convenience

In [2]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

# ---
vocab_size = 276 # the desired final vocabulary size
num_merges = vocab_size - 256
ids = list(tokens) # copy so we don't destory the original list

merges = {} # (int, int) --> int
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (32, 116) into a new token 256
merging (105, 110) into a new token 257
merging (101, 32) into a new token 258
merging (115, 32) into a new token 259
merging (111, 110) into a new token 260
merging (97, 110) into a new token 261
merging (116, 32) into a new token 262
merging (257, 103) into a new token 263
merging (101, 114) into a new token 264
merging (97, 116) into a new token 265
merging (101, 115) into a new token 266
merging (256, 104) into a new token 267
merging (97, 115) into a new token 268
merging (263, 32) into a new token 269
merging (97, 114) into a new token 270
merging (105, 260) into a new token 271
merging (100, 32) into a new token 272
merging (114, 111) into a new token 273
merging (111, 102) into a new token 274
merging (267, 258) into a new token 275


In [3]:
print("tokens length", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length 1230
ids length: 918
compression ratio: 1.34X


In [21]:
vocab = {idx: bytes([idx]) for idx in range(256)}
# and now also going up the merge tree
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1] # will return int???
    
def decode(ids):
    # given ids (list of integers), return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

In [5]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break # nothing else can be merged
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode("hello world!"))

[104, 101, 108, 108, 111, 32, 119, 111, 114, 108, 100, 33]
